# 05 - Evaluation: Stage 1 vs Stage 2 (RL)
Compare the pre-trained model against the RL fine-tuned model on the test set.

## Setup Instructions
1. Both Stage 1 and Stage 2 training must be complete
2. Checkpoints must exist at `MyDrive/CiteMind/checkpoints/`
3. Set runtime to **A100 GPU**
4. Run all cells in order

In [ ]:
# ── Step 1: Mount Google Drive ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR   = '/content/drive/MyDrive/CiteMind'
PRETRAIN_CKPT = f'{DRIVE_DIR}/checkpoints/pretrain/checkpoint_best.pt'
RL_CKPT       = f'{DRIVE_DIR}/checkpoints/rl/checkpoint_best_rl.pt'
EVAL_OUTPUT   = f'{DRIVE_DIR}/evaluation_results.json'

import os
assert os.path.exists(PRETRAIN_CKPT), f'Stage 1 checkpoint not found: {PRETRAIN_CKPT}'
assert os.path.exists(RL_CKPT), f'Stage 2 checkpoint not found: {RL_CKPT}'

# Show which RL checkpoint is loaded
import torch
rl_ckpt_info = torch.load(RL_CKPT, map_location='cpu', weights_only=False)
print('Drive mounted.')
print(f'Stage 1 checkpoint : OK')
print(f'Stage 2 checkpoint : OK  (step={rl_ckpt_info["step"]}, best_reward={rl_ckpt_info["best_reward"]:.4f})')
del rl_ckpt_info

In [ ]:
# ── Step 2: Clone / pull repo ────────────────────────────────────────────
import os
if not os.path.exists('/content/repo'):
    !git clone https://github.com/mohamedzait20003/ECE595NLP-Project /content/repo
else:
    !git -C /content/repo pull origin main
%cd /content/repo
print('Repo ready.')

In [ ]:
# ── Step 3: Install dependencies ─────────────────────────────────────────
!apt-get install -q libsndfile1
!pip install -q -r requirements.txt
!pip install -q torch --index-url https://download.pytorch.org/whl/cu124
!pip install -q sentence-transformers
print('Dependencies installed.')

In [ ]:
# ── Step 4: Extract data ─────────────────────────────────────────────────
import os, json, re
from pathlib import Path

DATA_ZIP = f'{DRIVE_DIR}/data.zip'

if not os.path.exists(DATA_ZIP):
    raise FileNotFoundError(
        f'Data zip not found at {DATA_ZIP}\n'
        'Please upload data.zip to MyDrive/CiteMind/ in Google Drive.'
    )

print(f'Found: {DATA_ZIP}')
!unzip -q -o "{DATA_ZIP}" -d /content/repo/src/data
print('Zip extracted.')

# Patch Windows absolute paths in manifest JSON files
AUDIO_BASE = '/content/repo/src/data/audio'
target = Path('/content/repo/src/data')

for manifest_name in ['train_manifest.json', 'val_manifest.json', 'test_manifest.json']:
    manifest_path = target / 'audio' / manifest_name
    if not manifest_path.exists():
        continue
    with open(manifest_path, 'r', encoding='utf-8') as f:
        entries = json.load(f)
    patched = 0
    for entry in entries:
        ap = entry.get('audio_path', '')
        if not ap.startswith('/content'):
            parts = re.split(r'[/\\\\]', ap)
            fname  = parts[-1]
            subdir = parts[-2] if len(parts) >= 2 else manifest_name.split('_')[0]
            entry['audio_path'] = f'{AUDIO_BASE}/{subdir}/{fname}'
            patched += 1
    with open(manifest_path, 'w', encoding='utf-8') as f:
        json.dump(entries, f)
    print(f'  Patched {patched} paths in {manifest_name}')

TEST_MANIFEST = '/content/repo/src/data/audio/test_manifest.json'
assert os.path.exists(TEST_MANIFEST), f'Test manifest not found: {TEST_MANIFEST}'
with open(TEST_MANIFEST, 'r') as f:
    n_test = len(json.load(f))
print(f'\nTest samples: {n_test}')

In [ ]:
# ── Step 5: Verify GPU ───────────────────────────────────────────────────
import sys, torch
sys.path.insert(0, '/content/repo')

assert torch.cuda.is_available(), 'No GPU found! Set runtime to A100.'
print(f'Device : {torch.cuda.get_device_name(0)}')
print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Step 6: Run evaluation on both checkpoints ──────────────────────────
from src.main.evaluation.evaluate import compare_checkpoints

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Use do_sample=True to match RL training conditions (temperature=0.7 for less noise than 0.9)
results = compare_checkpoints(
    checkpoint_paths={
        'Stage 1 (Pre-train)': PRETRAIN_CKPT,
        'Stage 2 (RL)':        RL_CKPT,
    },
    test_manifest_path=TEST_MANIFEST,
    device=device,
    max_samples=100,  # Use 100 samples for faster eval; set 0 for full test set
    do_sample=True,
    temperature=0.7,
    output_path=EVAL_OUTPUT,
)

In [ ]:
# ── Step 7: Comparison bar chart ─────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

metric_names = ['bleu', 'rouge_l', 'exact_match', 'author_accuracy', 'year_accuracy', 'format_accuracy']
display_names = ['BLEU', 'ROUGE-L', 'Exact Match', 'Author Acc', 'Year Acc', 'Format Acc']

stage1_vals = [results['Stage 1 (Pre-train)']['averages'][m] for m in metric_names]
stage2_vals = [results['Stage 2 (RL)']['averages'][m] for m in metric_names]

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, stage1_vals, width, label='Stage 1 (Pre-train)', color='#4C72B0')
bars2 = ax.bar(x + width/2, stage2_vals, width, label='Stage 2 (RL)', color='#DD8452')

ax.set_ylabel('Score')
ax.set_title('CiteMind: Stage 1 vs Stage 2 (RL) Evaluation')
ax.set_xticks(x)
ax.set_xticklabels(display_names)
ax.legend()
ax.set_ylim(0, 1.0)
ax.grid(axis='y', alpha=0.3)

# Add value labels on bars
for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.3f}',
            ha='center', va='bottom', fontsize=8)
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., h + 0.01, f'{h:.3f}',
            ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# ── Step 8: Side-by-side prediction samples ──────────────────────────────
import pandas as pd

n_show = 20
s1_preds = results['Stage 1 (Pre-train)']['predictions'][:n_show]
s2_preds = results['Stage 2 (RL)']['predictions'][:n_show]

rows = []
for i, (p1, p2) in enumerate(zip(s1_preds, s2_preds)):
    rows.append({
        '#': i + 1,
        'Reference': p1['reference'],
        'Stage 1': p1['generated'],
        'Stage 2 (RL)': p2['generated'],
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# ── Step 9: Per-metric distribution (box plot) ──────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for idx, (metric, display) in enumerate(zip(metric_names, display_names)):
    s1_scores = [s[metric] for s in results['Stage 1 (Pre-train)']['per_sample']]
    s2_scores = [s[metric] for s in results['Stage 2 (RL)']['per_sample']]

    bp = axes[idx].boxplot(
        [s1_scores, s2_scores],
        labels=['Stage 1', 'Stage 2 (RL)'],
        patch_artist=True,
        widths=0.5,
    )
    bp['boxes'][0].set_facecolor('#4C72B0')
    bp['boxes'][1].set_facecolor('#DD8452')

    axes[idx].set_title(display)
    axes[idx].set_ylim(-0.05, 1.05)
    axes[idx].grid(axis='y', alpha=0.3)

plt.suptitle('Per-Sample Score Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ── Step 10: Summary ─────────────────────────────────────────────────────
print('='*60)
print('EVALUATION SUMMARY')
print('='*60)

s1_avg = results['Stage 1 (Pre-train)']['averages']
s2_avg = results['Stage 2 (RL)']['averages']

print(f'\n{"Metric":20s}  {"Stage 1":>10s}  {"Stage 2":>10s}  {"Delta":>10s}')
print('-' * 56)
for m, d in zip(metric_names, display_names):
    v1 = s1_avg[m]
    v2 = s2_avg[m]
    delta = v2 - v1
    arrow = '+' if delta > 0 else ''
    print(f'{d:20s}  {v1:>10.4f}  {v2:>10.4f}  {arrow}{delta:>9.4f}')

print(f'\nResults saved to: {EVAL_OUTPUT}')